# Noesis — unit economics e inteligencia interna

## TL;DR

Noesis no debe entrenar un modelo fundacional en el MVP. Debe resolver localmente hechos, cálculos, clasificación y redacción repetible; usar Qwen local o compatible para lenguaje libre y reservar Haiku como respaldo. Con los supuestos base, los precios recomendados son **29 / 49 / 99 € + IVA** si Premium mantiene 100 minutos de voz. El punto de equilibrio operativo baja de unas 146 a 120 cuentas con un opex fijo supuesto de 3.500 €/mes.


## Contexto y método

El modelo separa: (1) COGS de software, (2) soporte y onboarding, (3) infraestructura fija y (4) opex corporativo para break-even. Los precios se tratan como importes **sin IVA**; también se calcula el daño de venderlos como IVA incluido. Los volúmenes por plan son supuestos editables que deben reemplazarse por telemetría del piloto.


In [1]:
from math import ceil
import pandas as pd

plans = ['Autónomo', 'Negocio', 'Premium']
current_price = [29, 39, 79]
recommended_price = [29, 49, 99]
mix = [0.55, 0.35, 0.10]
credits = [75, 300, 1500]
expected_use = [0.35, 0.40, 0.25]
internal_resolution = [0.60, 0.60, 0.60]
wa_utility = [30, 80, 200]
audio_minutes = [15, 60, 200]
documents = [15, 50, 200]
storage_gb = [0.25, 1, 3]
voice_minutes = [0, 0, 100]
support_minutes = [12, 20, 45]
onboarding_minutes = [30, 60, 120]

eur_per_usd = 1 / 1.1405
stripe_pct = 0.015 + 0.007
stripe_fixed = 0.25
wa_utility_rate = 0.0166
haiku_usd = 0.014       # 8k input + 1.2k output
qwen_usd = 0.003028     # mismo supuesto de tokens
qwen_share = 0.80
whisper_usd_hour = 0.04
external_doc_eur = 0.012
external_doc_share = 0.20
storage_usd_gb = 0.015
voice_usd_min = 0.11
voice_number_usd = 2
support_eur_hour = 25
fixed_platform = 65
fixed_opex = 3500
vat = 0.21
allocation_accounts = 100


In [2]:
weighted_external_eur = (qwen_share*qwen_usd + (1-qwen_share)*haiku_usd) * eur_per_usd

def plan_economics(i, price):
    stripe = price*stripe_pct + stripe_fixed
    wa = wa_utility[i]*wa_utility_rate
    ai_calls = credits[i]*expected_use[i]*(1-internal_resolution[i])
    ai = ai_calls*weighted_external_eur
    transcription = audio_minutes[i]/60*whisper_usd_hour*eur_per_usd
    extraction = documents[i]*external_doc_share*external_doc_eur
    storage = storage_gb[i]*storage_usd_gb*eur_per_usd
    voice = (voice_minutes[i]*voice_usd_min + voice_number_usd)*eur_per_usd if voice_minutes[i] else 0
    software_cogs = stripe + wa + ai + transcription + extraction + storage + voice
    support = support_minutes[i]/60*support_eur_hour
    onboarding = onboarding_minutes[i]/60*support_eur_hour/12
    fixed_share = fixed_platform/allocation_accounts
    contribution = price-software_cogs-support-onboarding-fixed_share
    return {
        'Precio': price, 'Stripe': stripe, 'WhatsApp': wa, 'IA': ai,
        'Audio': transcription, 'Documentos': extraction, 'Storage': storage,
        'Voz': voice, 'COGS software': software_cogs,
        'Margen bruto software': (price-software_cogs)/price,
        'Soporte': support, 'Onboarding': onboarding, 'Contribución': contribution,
        'Margen contribución': contribution/price,
        'Contribución si IVA incluido': price/(1+vat)-software_cogs-support-onboarding-fixed_share,
        'IA peor caso': credits[i]*haiku_usd*eur_per_usd,
    }

current = pd.DataFrame([plan_economics(i, p) for i,p in enumerate(current_price)], index=plans)
recommended = pd.DataFrame([plan_economics(i, p) for i,p in enumerate(recommended_price)], index=plans)
display(recommended.round(3))

             Precio  Stripe  WhatsApp     IA  Audio  Documentos  Storage     Voz  COGS software  Margen bruto software  Soporte  Onboarding  Contribución  Margen contribución  Contribución si IVA incluido  IA peor caso
Autónomo         29   0.888     0.498  0.048  0.009       0.036    0.003   0.000          1.482                  0.949    5.000       1.042        20.826                0.718                        15.793         0.921
Negocio          49   1.328     1.328  0.220  0.035       0.120    0.013   0.000          3.044                  0.938    8.333       2.083        34.889                0.712                        26.385         3.683
Premium          99   2.428     3.320  0.687  0.117       0.480    0.039  11.399         18.470                  0.813   18.750       4.167        56.964                0.575                        39.782        18.413


In [3]:
def scale_case(accounts, frame):
    revenue = accounts*sum(mix[i]*frame.iloc[i]['Precio'] for i in range(3))
    cogs = accounts*sum(mix[i]*frame.iloc[i]['COGS software'] for i in range(3))
    contribution = accounts*sum(mix[i]*frame.iloc[i]['Contribución'] for i in range(3))
    return revenue, cogs, contribution, contribution-fixed_opex

rows=[]
for n in [10, 50, 100, 500]:
    a=scale_case(n,current); r=scale_case(n,recommended)
    rows.append([n,*a,*r])
scale = pd.DataFrame(rows, columns=['Cuentas','Ingresos actuales','COGS actuales','Contribución actual','Resultado actual','Ingresos recom.','COGS recom.','Contribución recom.','Resultado recom.'])
display(scale.round(0))
blended_current = sum(mix[i]*current.iloc[i]['Contribución'] for i in range(3))
blended_recommended = sum(mix[i]*recommended.iloc[i]['Contribución'] for i in range(3))
print('Break-even actual:', ceil(fixed_opex/blended_current), 'cuentas')
print('Break-even recomendado:', ceil(fixed_opex/blended_recommended), 'cuentas')

   Cuentas  Ingresos actuales  COGS actuales  Contribución actual  Resultado actual  Ingresos recom.  COGS recom.  Contribución recom.  Resultado recom.
0       10              375.0           36.0                240.0           -3260.0            430.0         37.0                294.0           -3206.0
1       50             1875.0          180.0               1199.0           -2301.0           2150.0        186.0               1468.0           -2032.0
2      100             3750.0          361.0               2398.0           -1102.0           4300.0        373.0               2936.0            -564.0
3      500            18750.0         1803.0              11992.0            8492.0          21500.0       1864.0              14681.0           11181.0
Break-even actual: 146 cuentas
Break-even recomendado: 120 cuentas


In [4]:
gpu_month_usd = 0.58*730
comparison = pd.DataFrame({
    'Coste EUR/interacción': [0, 0.000810*eur_per_usd, qwen_usd*eur_per_usd, haiku_usd*eur_per_usd],
    '75/mes': [0, 75*0.000810*eur_per_usd, 75*qwen_usd*eur_per_usd, 75*haiku_usd*eur_per_usd],
    '300/mes': [0, 300*0.000810*eur_per_usd, 300*qwen_usd*eur_per_usd, 300*haiku_usd*eur_per_usd],
    '1500/mes': [0, 1500*0.000810*eur_per_usd, 1500*qwen_usd*eur_per_usd, 1500*haiku_usd*eur_per_usd],
}, index=['Cerebro Noesis','Cloudflare Qwen','Groq Qwen','Haiku'])
display(comparison.round(3))
print('GPU 16 GB siempre encendida:', round(gpu_month_usd*eur_per_usd,2), 'EUR/mes')
print('Cruce GPU vs Haiku:', round(gpu_month_usd/haiku_usd), 'interacciones/mes')
print('Cruce GPU vs Groq Qwen:', round(gpu_month_usd/qwen_usd), 'interacciones/mes')

                 Coste EUR/interacción  75/mes  300/mes  1500/mes
Cerebro Noesis                   0.000   0.000    0.000     0.000
Cloudflare Qwen                  0.001   0.053    0.213     1.065
Groq Qwen                        0.003   0.199    0.796     3.982
Haiku                            0.012   0.921    3.683    18.413
GPU 16 GB siempre encendida: 371.24 EUR/mes
Cruce GPU vs Haiku: 30243 interacciones/mes
Cruce GPU vs Groq Qwen: 139828 interacciones/mes


## Resultados y decisiones

- El coste de texto generativo no es el principal problema del margen: pesan más soporte, voz Premium y el precio neto de IVA.
- El cerebro interno protege margen y fiabilidad porque resuelve lo repetible con coste marginal casi cero y sin enviar datos fuera.
- **29 € + IVA** es sostenible para Autónomo. **39 €** deja Negocio demasiado cerca de Autónomo para el soporte prometido; se recomienda **49 € + IVA**.
- Premium debería ser **99 € + IVA** si incluye 100 minutos de voz; alternativa: **79 € + IVA sin voz** y un complemento de voz de al menos 15 €/mes.
- No se recomienda una GPU siempre encendida en el piloto: el coste puro solo cruza Haiku alrededor de 30.000 interacciones avanzadas al mes y Groq Qwen cerca de 140.000.


## Fuentes y limitaciones

Fuentes consultadas el 15/07/2026: [Stripe](https://stripe.com/es/pricing), [Anthropic Haiku](https://www.anthropic.com/claude/haiku), [Railway](https://docs.railway.com/pricing), [Resend](https://resend.com/docs/knowledge-base/what-is-resend-pricing), [WhatsApp Business](https://whatsappbusiness.com/products/platform-pricing/), [ECB](https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/index.en.html), [Retell](https://www.retellai.com/pricing), [Groq Speech-to-Text](https://console.groq.com/docs/speech-to-text), [Runpod](https://www.runpod.io/product/serverless) y [Qwen3](https://github.com/QwenLM/Qwen3).

Las tarifas cambian y pueden no incluir impuestos. WhatsApp utility España usa una rate card publicada por un tercero porque la tabla oficial dinámica no expuso el valor directamente durante la consulta. Soporte, volumen, extracción documental, mix, CAC y opex son hipótesis, no históricos. Deben recalibrarse tras 30 días de piloto.
